In [1]:
from pathlib import Path
import zipfile
import pandas as pd
import numpy as np

In [2]:
# Clone our BMEN 600 group's GitHub repository into Colab.
!git clone https://github.com/chelseataylor1/BMEN-600-Group-19.git

Cloning into 'BMEN-600-Group-19'...
remote: Enumerating objects: 256, done.
remote: Counting objects: 100% (256/256), done.
remote: Compressing objects: 100% (238/238), done.
remote: Total 256 (delta 86), reused 5 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (256/256), 133.33 MiB | 23.51 MiB/s, done.
Resolving deltas: 100% (86/86), done.


In [3]:
# Locate the ZIP file downloaded from GitHub.
zip_path = Path("BMEN-600-Group-19/data/processed/OutputFromNotebook2_EHG_Dataset.zip")

# Extract the dataset.
with zipfile.ZipFile(zip_path, "r") as zip_file:
    zip_file.extractall("/content")

# Define the locations of the extracted files.
dataset_path = Path("/content/BMEN600_Matched_Trimmed_EHG_Dataset")
recordings_path = dataset_path / "recordings"
metadata_path = dataset_path / "matched_pregnancies.csv"

print("Dataset extracted successfully!")
print("Recordings:", recordings_path)
print("Metadata:", metadata_path)

Dataset extracted successfully!
Recordings: /content/BMEN600_Matched_Trimmed_EHG_Dataset/recordings
Metadata: /content/BMEN600_Matched_Trimmed_EHG_Dataset/matched_pregnancies.csv


In [4]:
# Load the Matched Pregnancy Metadata.
df_matched = pd.read_csv(metadata_path)

# Confirm the metadata loaded successfully.
print("Matched pregnancies loaded:", len(df_matched))

# Preview the metadata.
display(df_matched.head())

Matched pregnancies loaded: 35


,pregnancy_id,age,early_id,early_week,weight_early,later_id,later_week,weight_later,height,placenta,gestation,newborn_weight,outcome,weeks_between
0,17,22,620,23.9,73,1529,31.9,81,168,end,41.1,3930,induced,8.0
1,18,38,656,22.4,77,1385,31.1,82,167,end,41.3,3710,induced,8.7
2,19,26,665,23.1,83,664,30.4,84,165,front,37.1,2860,induced,7.3
3,20,32,757,22.4,68,756,30.6,72,162,end,39.7,3500,induced,8.2
4,21,29,774,25.0,77,775,31.1,83,171,front,41.6,3180,induced,6.1


In [5]:
# Verify the trimmed EHG recordings
# Update recordings_path to where the .npz files are located
recordings_path = Path("BMEN-600-Group-19/BMEN600_Matched_Trimmed_EHG_Dataset/recordings")

# Find all trimmed EHG recordings.
recording_files = sorted(recordings_path.glob("*.npz"))

# Confirm the number of recordings.
print("Trimmed EHG recordings loaded:", len(recording_files))

# Preview the contents of one recording if found.
if len(recording_files) > 0:
    with np.load(recording_files[0]) as recording:
        print("Recording:", recording_files[0].name)
        print("Available variables:", recording.files)
        print("Sampling frequency:", recording["fs"])
        print("Number of samples:", len(recording["trimmed_S4"]))
else:
    print("No .npz files found in the recordings path:", recordings_path)

Trimmed EHG recordings loaded: 70
Recording: icehg1010.npz
Available variables: ['trimmed_time', 'trimmed_S1', 'trimmed_S2', 'trimmed_S3', 'trimmed_S4', 'fs']
Sampling frequency: 20
Number of samples: 29199


In [6]:
"""
Median frequency (MF) of EHG channels S1-S4 for every matched pregnancy
with outcome = "induced".

For each pregnancy the early (~23 wk) and later (~31 wk) recordings are
loaded, and MF is computed for S1, S2, S3 and S4. Output: one table with a
row per pregnancy (early MF, later MF and the change later - early).

Method: each trimmed, filtered recording is converted to a power spectrum
(whole recording, no windows), and the median frequency is the frequency that
splits the power between 0.3 and 3 Hz into two equal halves. The signals are
used as provided (no extra filtering here).

Change columns:
  *_change          = later MF - early MF (Hz)
  *_change_per_week = *_change / weeks_between (Hz per week), where
                      weeks_between = later_week - early_week for that pair

Packages: numpy, pandas, scipy
"""

import os
import numpy as np
import pandas as pd
from scipy.signal import periodogram

# ----------------------------------------------------------------------------
# SETTINGS - updated to the correct repository path containing the npz files
# ----------------------------------------------------------------------------
DATA_DIR = "BMEN-600-Group-19/BMEN600_Matched_Trimmed_EHG_Dataset"
OUTCOME = "induced"            # induced | cesarean | induced-cesarean
CHANNELS = ["S1", "S2", "S3", "S4"]

BAND = (0.3, 3.0)      # Hz, band in which the median frequency is calculated

OUT_CSV = f"median_frequency_{OUTCOME}.csv"


def power_spectrum(x, fs):
    """Power spectrum of the whole trimmed, filtered recording (one FFT-based
    periodogram, no windowing into segments; Hann taper, mean removed)."""
    x = np.asarray(x, dtype=float)
    return periodogram(x, fs=fs, window="hann", detrend="constant",
                       scaling="density")


def median_frequency(x, fs, band=BAND):
    """Frequency (Hz) that splits the power spectrum within `band` into two
    halves of equal power (linear interpolation of the cumulative power)."""
    freqs, psd = power_spectrum(x, fs)

    keep = (freqs >= band[0]) & (freqs <= band[1])
    freqs, psd = freqs[keep], psd[keep]

    cum = np.cumsum(psd)
    cum = cum / cum[-1]
    return float(np.interp(0.5, cum, freqs))


def load_recording(rec_id):
    path = os.path.join(DATA_DIR, "recordings", f"icehg{int(rec_id)}.npz")
    return np.load(path)


def mf_for_recording(rec_id):
    d = load_recording(rec_id)
    fs = float(d["fs"])
    results = {}
    for ch in CHANNELS:
        sig = d[f"trimmed_{ch}"]
        results[ch] = median_frequency(sig, fs)
    return results

# Calculating the Change in Median Frequency per Weekly Recordings: 23 & 31 Weeks (Jerold)

**Change in Median Frequency between early (23 weeks) & later (31 weeks) EHG recordings across individual pregnancies**

In [7]:
# 1. Filter for the target outcome group and remove pregnancy_id 1393
df_group = df_matched[df_matched["outcome"] == "induced"].copy()
df_group = df_group[df_group["pregnancy_id"] != 1393]

individual_results = []

for _, row in df_group.iterrows():
    preg_id = int(row["pregnancy_id"])
    weeks_between = float(row["weeks_between"])

    # Load and calculate early (~23 wk) and later (~31 wk) MF for channels S1-S4
    early_mf = mf_for_recording(row["early_id"])
    later_mf = mf_for_recording(row["later_id"])

    preg_data = {
        "pregnancy_id": preg_id,
        "early_week": row["early_week"],
        "later_week": row["later_week"],
        "weeks_between": weeks_between,
    }

    # Compute absolute change and weekly rate per channel
    for ch in CHANNELS:
        change = later_mf[ch] - early_mf[ch]
        preg_data[f"{ch}_early_MF"] = round(early_mf[ch], 4)
        preg_data[f"{ch}_later_MF"] = round(later_mf[ch], 4)
        preg_data[f"{ch}_change"] = round(change, 4)
        preg_data[f"{ch}_change_per_week"] = round(
            change / weeks_between if weeks_between > 0 else 0.0, 4
        )

    individual_results.append(preg_data)

# 2. Convert to DataFrame and display individual results
df_induced_mf = pd.DataFrame(individual_results)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
display(df_induced_mf)


,pregnancy_id,early_week,later_week,weeks_between,S1_early_MF,S1_later_MF,S1_change,S1_change_per_week,S2_early_MF,S2_later_MF,S2_change,S2_change_per_week,S3_early_MF,S3_later_MF,S3_change,S3_change_per_week,S4_early_MF,S4_later_MF,S4_change,S4_change_per_week
0,17,23.9,31.9,8.0,0.4747,0.3487,-0.1260,-0.0158,0.3487,0.3914,0.0428,0.0053,0.4429,0.4394,-0.0035,-0.0004,0.3641,0.3544,-0.0097,-0.0012
1,18,22.4,31.1,8.7,1.3358,0.3722,-0.9636,-0.1108,0.4551,0.3675,-0.0876,-0.0101,0.6435,0.3547,-0.2888,-0.0332,0.4768,0.3492,-0.1276,-0.0147
2,19,23.1,30.4,7.3,0.4552,1.1061,0.6509,0.0892,0.4375,0.4917,0.0542,0.0074,0.4427,0.8185,0.3758,0.0515,0.4262,0.5412,0.1150,0.0158
3,20,22.4,30.6,8.2,0.3276,0.3718,0.0442,0.0054,0.3154,0.3443,0.0289,0.0035,0.3275,0.3657,0.0382,0.0047,0.3146,0.3527,0.0381,0.0046
4,21,25.0,31.1,6.1,0.5884,0.7737,0.1853,0.0304,0.3992,0.3979,-0.0013,-0.0002,0.5112,0.8541,0.3429,0.0562,0.4176,0.3997,-0.0179,-0.0029
5,22,22.4,30.1,7.7,0.5196,0.4112,-0.1084,-0.0141,0.4306,0.4042,-0.0264,-0.0034,0.7492,0.4279,-0.3213,-0.0417,0.4333,0.4095,-0.0239,-0.0031
6,23,21.4,30.4,9.0,0.3522,0.3668,0.0146,0.0016,0.3522,0.3672,0.0150,0.0017,0.3522,0.5733,0.2211,0.0246,0.3521,0.4271,0.0751,0.0083
7,24,22.3,30.3,8.0,1.1226,0.5357,-0.5869,-0.0734,0.4495,0.4433,-0.0062,-0.0008,0.4954,0.5304,0.0349,0.0044,0.4755,0.4387,-0.0367,-0.0046
8,25,22.4,30.6,8.2,0.3297,0.4527,0.1230,0.0150,0.4113,0.4967,0.0854,0.0104,0.3847,0.4837,0.0990,0.0121,0.3350,0.5487,0.2137,0.0261
9,26,23.1,30.9,7.8,0.5840,0.5601,-0.0239,-0.0031,0.5414,0.5564,0.0150,0.0019,0.5758,0.5605,-0.0153,-0.0020,0.5483,0.5585,0.0102,0.0013


**Change in Median Frequency between early (23 weeks) & later (31 weeks) EHG recordings across ALL pregnancies**

In [8]:
# Define table using the calculated DataFrame
table = df_induced_mf

# Extract value columns for calculation
mf_cols = ['S1_early_MF', 'S1_later_MF', 'S1_change', 'S1_change_per_week',
           'S2_early_MF', 'S2_later_MF', 'S2_change', 'S2_change_per_week',
           'S3_early_MF', 'S3_later_MF', 'S3_change', 'S3_change_per_week',
           'S4_early_MF', 'S4_later_MF', 'S4_change', 'S4_change_per_week']

# Generate summary table for mean, median, and std
summary_stats = table[mf_cols].agg(['mean', 'median', 'std']).round(4)
print(summary_stats.T)

                      mean  median     std
S1_early_MF         0.6667  0.5121  0.3677
S1_later_MF         0.5829  0.4412  0.3327
S1_change          -0.0838 -0.0337  0.4402
S1_change_per_week -0.0087 -0.0044  0.0548
S2_early_MF         0.4818  0.4435  0.2251
S2_later_MF         0.4307  0.4166  0.0678
S2_change          -0.0510  0.0058  0.2279
S2_change_per_week -0.0071  0.0007  0.0322
S3_early_MF         0.5344  0.4834  0.2335
S3_later_MF         0.4997  0.4392  0.1529
S3_change          -0.0347 -0.0094  0.2250
S3_change_per_week -0.0041 -0.0012  0.0307
S4_early_MF         0.4661  0.4298  0.1624
S4_later_MF         0.4557  0.4080  0.1618
S4_change          -0.0104 -0.0117  0.1058
S4_change_per_week -0.0015 -0.0014  0.0130


**Based on your Colab notebook execution (Induced_medfreq.ipynb), yes, there is a measurable change in median frequency between the early (23 weeks) and later (31 weeks) recordings across the induced delivery cohort ($N = 22$).**

1. Overrall Decreasing Trend across S1 to S4 from week 23 to 31
2. Channel S1 Showing Largest Shift (Early Median S1: 0.5121 Hz, Later Median S1: 0.4412 Hz, Median Change S1: -0.0337 Hz)
3. Channels S2, S3, & S4 Show slight downard trend.

In [9]:
import shutil
from pathlib import Path
from google.colab import files

# 1. Copy the original dataset into a new folder
output_folder = Path("/content/BMEN600_MF_Dataset")
shutil.copytree(dataset_path, output_folder, dirs_exist_ok=True)

# 2. Add the new median frequency variables to the original metadata
df_updated = df_matched.merge(
    df_induced_mf,
    on="pregnancy_id",
    how="left"
)

# 3. Save the updated CSV
df_updated.to_csv(output_folder / "matched_pregnancies.csv", index=False)

# 4. Create the ZIP file
zip_path = shutil.make_archive(
    "/content/BMEN600_MF_Dataset",
    "zip",
    root_dir="/content",
    base_dir="BMEN600_MF_Dataset"
)

# 5. Download the ZIP
print("ZIP created successfully!")
files.download(zip_path)

ZIP created successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>